# Blind Side-Channel Analysis Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zer0Leak/CISSA/blob/main/Educational/SCA_articles/Rezaeezade_2025_blind_kyber/main.ipynb)&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
[![View on GitHub](https://img.shields.io/badge/View%20on-GitHub-black?logo=github)](https://github.com/Zer0Leak/CISSA/blob/main/Educational/SCA_articles/Rezaeezade_2025_blind_kyber/main.ipynb)

This notebook is a hands-on tutorial inspired by the paper *Breaking the Blindfold: Deep Learning-based Blind Side-channel Analysis*.

**Disclaimer:** I am **not** an author of the paper or the attack described in it. This notebook was prepared solely for educational and study purposes.

# Download original paper PDF and Datasets from here

[https://zenodo.org/records/15614359](https://zenodo.org/records/15614359)

## Reference

```bibtex
@misc{cryptoeprint:2025/157,
      author = {Azade Rezaeezade and Trevor Yap and Dirmanto Jap and Shivam Bhasin and Stjepan Picek},
      title = {Breaking the Blindfold: Deep Learning-based Blind Side-channel Analysis},
      howpublished = {Cryptology {ePrint} Archive, Paper 2025/157},
      year = {2025},
      url = {https://eprint.iacr.org/2025/157}
}
```

## Who Am I

[Edgard Lima](https://www.linkedin.com/in/edgardlima/)

### Past
- 10 years at CESAR (not continuous)
- 25+ years in industry/research
- Consultant at Nokia Research Center (NRC), 2007/2008 - Helsinki, Finland
- Author of GStreamer Video4Linux2 SRC
  - Linux webcam
- Programming since 1995
- Bachelor: Computer Science (2002), CIn UFPE
- Master: Algorithms and Distributed Systems (Distributed van Emde Boas tree)
- Device driver development
- Embedded systems development
- Customizing Linux kernel for new hardware
- Teacher: Operating Systems and Formal Languages (2017/2018)
- Dissected or developed several network protocols
  - CAN, J1939, Bluetooth (physical, L2CAP, SDP, RFCOMM), ...
- Compilers, sockets, scraping, IPC, concurrency, etc.
- C, C++, Python, Objective-C, Swift, Java
  - It is messy in my mind (I need an auto-complete now :-))

### Now
- PhD student at CESAR School: Cybersecurity
- Researcher at CISSA/CESAR: Cybersecurity
- Dad of two lovely kids, and owner of two fierce Dobermanns

## Context

This tutorial was prepared as part of the training activities of **CISSA @ CESAR**, with a focus on cybersecurity research.

- **CISSA:** [https://www.cesar.org.br/web/english/cissa](https://www.cesar.org.br/web/english/cissa)
- **CESAR:** [https://www.cesar.org.br/web/english](https://www.cesar.org.br/web/english)

<p align="left">
  <img src="extras/cissa.png" alt="CISSA at CESAR" width="360">
</p>


## Plan (AES First)

This notebook will implement the full pipeline for **AES first**.

**Kyber is intentionally left for later** to keep the tutorial more didactic. We will create a separate notebook for Kyber-specific details.

1. [Choose the target variables](#step-1-choose-aes-target-variables)
2. [Prepare the offline key-vs-variables theoretical joint distribution](#step-2-offline-theoretical-joint-distribution)
3. [Find PoIs (Points of Interest) for each variable](#step-3-finding-pois-for-h_m-and-h_y)
4. [Build labels from those PoIs](#step-4-build-labels-from-those-pois)
5. Train the DNN (with noisy labels handling in mind).
6. Build empirical distributions from predicted/assigned labels.
7. Score key candidates by comparing empirical vs theoretical distributions.

# Step 1: Choose AES Target Variables

We follow the paper's AES setting and attack **AES-128, one first-round key byte at a time**.

For a chosen byte index `b in {0, ..., 15}`:

- `m = plaintext[b]` is the public variable.
- `k* = key[b]` is the unknown secret byte we want to recover.
- `y = Sbox(m xor k*)` is the sensitive intermediate value.

This matches the paper: for AES, the relevant public/sensitive pair is the plaintext byte and the **first-round S-box output**, because the S-box is the first nonlinear operation and is still byte-wise.

Under a Hamming-weight leakage model, the variables used in the joint distribution are:

- `h_m_theoretical = HW(m)`
- `h_y_theoretical = HW(y)`

For each key guess `k in {0, ..., 255}`, we will later precompute the theoretical distribution `Pr((h_m_theoretical, h_y_theoretical) | k)` by iterating over all possible plaintext-byte values `m in {0, ..., 255}`.

Important notation for the rest of the notebook:

- `(h_m_theoretical, h_y_theoretical)` is the **theoretical** Hamming-weight pair induced by a key guess.
- `(h_m, h_y)` will be the **empirical** pair later assigned or predicted from traces.

The attack succeeds when the empirical distribution built from traces is closer to the correct key's theoretical distribution than to the wrong ones.

In [ ]:
# Core numerical dependency used throughout the tutorial.
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# AES-128 is attacked byte by byte, so the standard AES S-box is represented
# as a 256-entry lookup table indexed by an unsigned 8-bit value.
_aes_sbox = np.array([
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16,
], dtype=np.uint8)

# Precompute HW(x) for every byte x in [0, 255]. Reusing a lookup table keeps
# the code simple and avoids recomputing bit counts throughout the notebook.
_hw_lut = np.unpackbits(
    np.arange(256, dtype=np.uint8)[:, np.newaxis],
    axis=1,
).sum(axis=1).astype(np.uint8)


def aes_sbox(values):
    """Apply the AES S-box to one byte or to an array of bytes.

    Parameters
    ----------
    values : int or np.ndarray
        Byte value(s) in the range [0, 255].

    Returns
    -------
    np.ndarray or np.uint8
        S-box output with the same shape as ``values``.
    """
    values = np.asarray(values, dtype=np.uint8)
    return _aes_sbox[values]


def hamming_weight(values):
    """Return HW(x) for one byte or for an array of bytes.

    This tutorial uses the Hamming-weight leakage model, so this helper is
    reused when building both theoretical and empirical distributions.
    """
    values = np.asarray(values, dtype=np.uint8)
    return _hw_lut[values]

In [ ]:
# Example usage:

m_0 = 0xA7
k_0_guess = 0x4D
xored = m_0 ^ k_0_guess
print(f"XOR of plaintext byte and key guess: {xored:#04x}")
sbox_output = aes_sbox(xored)
print(f"S-box output: {sbox_output:#04x} = {bin(sbox_output)}")
hw = hamming_weight(sbox_output)
print(f"Hamming weight of S-box output: {hw}")
print(f"An array for values can also be processed: {hamming_weight(aes_sbox([0x01, 0xCC, 0xEF]))}")

## Step 2: Offline Theoretical Joint Distribution

We now implement **Algorithm 2** from the paper for AES.

For each key hypothesis `k in {0, ..., 255}` and each plaintext byte value `m in {0, ..., 255}`, we compute:

- `y = Sbox(m xor k)`
- `h_m_theoretical = HW(m)`
- `h_y_theoretical = HW(y)`

Then we count how often each pair `(h_m_theoretical, h_y_theoretical)` appears and normalize the counts to obtain a probability distribution.

In [ ]:
def build_aes_theoretical_joint_distribution():
    """Compute the AES theoretical joint distribution for all 256 key guesses.

    The returned tensors follow the paper's Algorithm 2:
    ``joint_counts[k, hm, hy]`` stores how many plaintext-byte values ``m``
    produce the pair ``(HW(m), HW(Sbox(m xor k)))`` for the key guess ``k``.
    ``joint_probabilities`` is the normalized version used later for scoring.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        ``joint_counts`` has shape ``(256, 9, 9)`` and dtype ``np.uint16``.
        ``joint_probabilities`` has shape ``(256, 9, 9)`` and dtype ``np.float64``.
    """
    plaintext_value_array = np.arange(256, dtype=np.uint8)
    h_m_theoretical_array = hamming_weight(plaintext_value_array)

    # Axes: key guess, HW(m), HW(y). Each Hamming weight ranges from 0 to 8.
    joint_counts = np.zeros((256, 9, 9), dtype=np.uint16)

    for key_guess in range(256):
        sbox_inputs = np.bitwise_xor(plaintext_value_array, np.uint8(key_guess))
        y_array = aes_sbox(sbox_inputs)
        h_y_theoretical_array = hamming_weight(y_array)

        # ``np.add.at`` accumulates repeated ``(h_m, h_y)`` pairs correctly. I.e. add 1 count for each plaintext value that produces the same HW pair.
        np.add.at(joint_counts[key_guess], (h_m_theoretical_array, h_y_theoretical_array), 1)

    joint_probabilities = joint_counts.astype(np.float64) / plaintext_value_array.size
    return joint_counts, joint_probabilities


aes_joint_counts, aes_joint_probabilities = build_aes_theoretical_joint_distribution()

# Basic sanity checks: for each key guess, 256 plaintext values are enumerated
# and the normalized distribution must sum to 1.
assert np.all(aes_joint_counts.sum(axis=(1, 2)) == 256)
assert np.allclose(aes_joint_probabilities.sum(axis=(1, 2)), 1.0)

print("Counts tensor shape:", aes_joint_counts.shape)
print("Probability tensor shape:", aes_joint_probabilities.shape)
random_key_guess = np.random.randint(0, 256)
print(f"Example: total probability mass for key guess 0x{random_key_guess:02X} =", aes_joint_probabilities[random_key_guess].sum())

## Visualizing Theoretical Joint Distributions

The tensor `aes_joint_probabilities` has shape `(256, 9, 9)`, so each key guess corresponds to one `9 x 9` heatmap.

The horizontal axis is `h_m_theoretical = HW(m)` and the vertical axis is `h_y_theoretical = HW(Sbox(m xor k))`.

We use a dark-to-bright color scale so low probabilities remain dark and high probabilities stand out clearly.

In [ ]:
def plot_aes_joint_probabilities(joint_probabilities, key_guesses, annotate=False):
    """Plot one or more AES theoretical joint distributions as heatmaps.

    Parameters
    ----------
    joint_probabilities : np.ndarray
        Array with shape ``(256, 9, 9)`` containing the theoretical
        distributions for every AES key hypothesis.
    key_guesses : int or list[int] or tuple[int, ...]
        One key guess or a collection of key guesses to display.
    annotate : bool, default=False
        If ``True``, print the probability of each cell on top of the heatmap.
        This is useful for a single plot, but visually busy for many subplots.

    Returns
    -------
    tuple[matplotlib.figure.Figure, np.ndarray]
        The created figure and axes, which can be further customized if needed.
    """
    key_guesses = np.atleast_1d(key_guesses).astype(int)

    if np.any((key_guesses < 0) | (key_guesses > 255)):
        raise ValueError("Every key guess must be in the range [0, 255].")

    ncols = len(key_guesses)
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(5.2 * ncols, 4.8),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)

    vmax = float(joint_probabilities[key_guesses].max())
    image = None

    for axis, key_guess in zip(axes, key_guesses):
        heatmap = joint_probabilities[key_guess]
        display_heatmap = heatmap.T
        image = axis.imshow(
            display_heatmap,
            origin="lower",
            cmap="inferno",
            vmin=0.0,
            vmax=vmax,
            interpolation="nearest",
            aspect="equal",
        )

        axis.set_title(f"AES theoretical joint distribution\nkey guess = 0x{key_guess:02X}")
        axis.set_xlabel(r"$h_m = HW(m)$")
        axis.set_ylabel(r"$h_y = HW(Sbox(m \oplus k))$")
        axis.set_xticks(range(9))
        axis.set_yticks(range(9))
        axis.set_xticks(np.arange(-0.5, 9, 1), minor=True)
        axis.set_yticks(np.arange(-0.5, 9, 1), minor=True)
        axis.grid(which="minor", color="white", linewidth=1.2)
        axis.tick_params(which="minor", bottom=False, left=False)

        if annotate:
            for h_m_index in range(9):
                for h_y_index in range(9):
                    probability = heatmap[h_m_index, h_y_index]
                    text_color = "black" if probability > (0.55 * vmax) else "white"
                    axis.text(
                        h_m_index,
                        h_y_index,
                        f"{probability:.3f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color=text_color,
                    )

    colorbar = fig.colorbar(image, ax=axes, shrink=0.88, pad=0.02)
    colorbar.set_label("Probability mass")
    return fig, axes


# Show a few representative key hypotheses side by side.
# These plots make it easier to see that each key guess induces a distinct
# theoretical distribution over the pair (HW(m), HW(y)).
example_key_guesses = [0x00, 0x2B, 0x7F]
fig, axes = plot_aes_joint_probabilities(
    aes_joint_probabilities,
    key_guesses=example_key_guesses,
    annotate=False,
)
plt.show()

# Step 3: Finding PoIs for `h_m` and `h_y`

We now move to **PoI selection**.

Important assumption: in the paper, this is the **only non-blind step**. To find PoIs, the attacker uses a calibration/profiling set for which the public plaintext byte and the attacked key byte are known for each trace. This allows us to compute the true target values `HW(m)` and `HW(y) = HW(Sbox(m xor k*))` and correlate them with the trace samples.

This means:

- plaintext is used here only to locate informative sample points
- plaintext is **not** needed later when we build the empirical distribution and rank key guesses

For AES, we compute two correlation traces:

- one correlation curve against `h_m = HW(m)`
- one correlation curve against `h_y = HW(Sbox(m xor k*))`

Then we select the sample indices with the largest absolute correlations. These indices are our PoIs.

The paper often uses **50 PoIs per variable**, so the helper functions below are written with that workflow in mind.

In [ ]:
def _extract_aes_byte(values, byte_index):
    """Extract one AES byte column from a 1D or 2D per-trace byte array.

    Parameters
    ----------
    values : np.ndarray
        Either a 1D array of shape ``(n_traces,)`` containing one byte per
        trace, or a 2D array of shape ``(n_traces, 16)`` containing full AES
        plaintexts, round states, or keys.
    byte_index : int
        Targeted AES byte index in ``{0, ..., 15}`` when ``values`` is 2D.

    Returns
    -------
    np.ndarray
        Array of shape ``(n_traces,)`` with dtype ``np.uint8``.
    """
    values = np.asarray(values, dtype=np.uint8)

    if values.ndim == 1:
        return values

    if values.ndim == 2:
        if not 0 <= byte_index < values.shape[1]:
            raise ValueError("byte_index is out of range for the provided byte array.")
        return values[:, byte_index]

    raise ValueError("values must be a 1D or 2D per-trace byte array.")


def pearson_correlation_to_trace_samples(traces, target_values):
    """
    Compute the Pearson correlation between a target vector and every sample
    position of a trace matrix.

    This is a vectorized implementation commonly used in CPA
    (Correlation Power Analysis). Each column of ``traces`` is treated as one
    time sample across all acquisitions, and correlated against the same
    ``target_values`` vector.

    Parameters
    ----------
    traces : array-like of shape (n_traces, n_samples)
        Matrix containing the measured traces.

        - Each row corresponds to one trace / acquisition.
        - Each column corresponds to one sample index (time point).

        Example
        -------
        ``traces[i, j]`` is the value of sample ``j`` in trace ``i``.

    target_values : array-like of shape (n_traces,)
        One target value per trace. In side-channel analysis this is often a
        hypothetical leakage value, such as:

        - Hamming weight of an intermediate value
        - Hamming distance
        - output of some leakage model for a given key guess

        In our case here, is the Hamming weight of either the plaintext byte or the S-box output for each trace.
        We are looking for sample points that are most correlated with these target values, which is the basis of PoI selection in CPA.

        The number of entries must match the number of rows in ``traces``.

    Returns
    -------
    np.ndarray of shape (n_samples,)
        Pearson correlation coefficient for each sample position.

        The returned vector can be interpreted as a correlation trace:
        ``result[j]`` is the correlation between ``target_values`` and the
        j-th sample column ``traces[:, j]``.

    Raises
    ------
    ValueError
        If ``traces`` is not 2-dimensional.
    ValueError
        If the number of traces does not match the number of target values.

    Notes
    -----
    Pearson correlation between two vectors x and y is:

    .. math::

        r = \\frac{\\sum_i (x_i - \\bar{x})(y_i - \\bar{y})}
                 {\\sqrt{\\sum_i (x_i - \\bar{x})^2}
                  \\sqrt{\\sum_i (y_i - \\bar{y})^2}}

    This function applies that formula to each column of ``traces`` against the
    same ``target_values`` vector, without using an explicit Python loop.

    Implementation details
    ----------------------
    1. Convert inputs to ``float64`` for numerical stability.
    2. Mean-center each trace sample column.
    3. Mean-center the target vector.
    4. Compute all numerators at once using matrix multiplication.
    5. Compute all denominators in vectorized form.
    6. Divide safely, returning 0 where the denominator is zero.

    Zero-variance behavior
    ----------------------
    If a trace sample column is constant across all traces, or if
    ``target_values`` is constant, the corresponding denominator becomes zero.
    In that case this function returns 0 for that sample instead of NaN or Inf.

    Examples
    --------
    Basic usage:

    >>> traces = np.array([
    ...     [1.0, 2.0, 3.0],
    ...     [2.0, 1.0, 4.0],
    ...     [3.0, 0.0, 5.0],
    ...     [4.0, 1.0, 6.0],
    ... ])
    >>> target_values = np.array([10.0, 20.0, 30.0, 40.0])
    >>> pearson_correlation_to_trace_samples(traces, target_values)
    array([ 1.        , -0.63245553,  1.        ])

    Typical CPA usage:

    >>> # target_values could be something like:
    >>> # HW(SBOX[plaintext_byte ^ key_guess]) for each trace
    >>> corr = pearson_correlation_to_trace_samples(traces, target_values)
    >>> peak = np.max(np.abs(corr))

    See Also
    --------
    numpy.corrcoef : General-purpose correlation computation.
    """
    # Convert inputs to NumPy arrays with float64 precision.
    # This ensures the math below uses floating-point operations even if the
    # caller passes lists or integer arrays.
    traces = np.asarray(traces, dtype=np.float64)
    target_values = np.asarray(target_values, dtype=np.float64).reshape(-1)

    # We expect a 2D trace matrix:
    #   axis 0 -> trace index
    #   axis 1 -> sample index
    if traces.ndim != 2:
        raise ValueError("traces must have shape (n_traces, n_samples).")

    # Each trace must have exactly one corresponding target value.
    if traces.shape[0] != target_values.shape[0]:
        raise ValueError("The number of traces must match the number of target values.")

    # Center each sample column independently:
    # subtract the mean of each column from that column.
    #
    # Result:
    #   centered_traces[:, j] = traces[:, j] - mean(traces[:, j])
    #
    # keepdims=True keeps the mean shape as (1, n_samples), which makes the
    # broadcasting explicit and easier to reason about.
    centered_traces = traces - traces.mean(axis=0, keepdims=True)

    # Center the target vector:
    #   centered_target[i] = target_values[i] - mean(target_values)
    centered_target = target_values - target_values.mean()

    # Compute the Pearson numerator for all sample columns at once.
    #
    # centered_traces.T has shape (n_samples, n_traces)
    # centered_target   has shape (n_traces,)
    #
    # So the matrix-vector multiplication produces shape (n_samples,).
    #
    # For each sample j, this is:
    #   numerator[j] = sum_i centered_traces[i, j] * centered_target[i]
    numerator = centered_traces.T @ centered_target

    # Compute the denominator for each sample column.
    #
    # sum(centered_traces ** 2, axis=0)
    #     -> squared norm of each centered sample column, shape (n_samples,)
    #
    # sum(centered_target ** 2)
    #     -> squared norm of the centered target vector, scalar
    #
    # The final denominator is:
    #   sqrt( ||column_j||^2 * ||target||^2 )
    denominator = np.sqrt(
        np.sum(centered_traces ** 2, axis=0) * np.sum(centered_target ** 2)
    )

    # Perform safe division.
    #
    # If denominator[j] == 0, Pearson correlation is undefined for that sample
    # (for example, due to zero variance). We choose to return 0.0 there.
    correlation = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=np.float64),
        where=denominator > 0,
    )

    return correlation


def select_top_pois(correlation_trace, num_pois=50, min_separation=1):
    """Select PoIs with largest absolute correlation.

    Parameters
    ----------
    correlation_trace : np.ndarray
        Correlation value for each sample point.
    num_pois : int, default=50
        Number of PoIs to return.
    min_separation : int, default=1
        Minimum distance in samples between two selected PoIs. A value of 1
        only prevents duplicates; larger values enforce spacing.

    Returns
    -------
    np.ndarray
        Sorted sample indices of the selected PoIs.
    """
    correlation_trace = np.asarray(correlation_trace, dtype=np.float64).reshape(-1)

    if num_pois <= 0:
        raise ValueError("num_pois must be a positive integer.")

    if min_separation <= 0:
        raise ValueError("min_separation must be a positive integer.")

    ranked_indices = np.argsort(np.abs(correlation_trace))[::-1]
    selected_indices = []

    for sample_index in ranked_indices:
        if all(abs(int(sample_index) - chosen_index) >= min_separation for chosen_index in selected_indices):
            selected_indices.append(int(sample_index))
            if len(selected_indices) == num_pois:
                break

    return np.array(sorted(selected_indices), dtype=np.int64)


def _prepare_aes_key_bytes(key_bytes, byte_index, num_traces):
    """Normalize AES key input for PoI selection.

    Parameters
    ----------
    key_bytes : int or np.ndarray
        AES key information provided either as:

        - one scalar byte reused for every trace,
        - a 1D array of shape ``(n_traces,)`` with the attacked key byte for
          each trace, or
        - a 2D array of shape ``(n_traces, 16)`` with full AES keys.
    byte_index : int
        Targeted AES byte index when full AES keys are provided.
    num_traces : int
        Number of traces expected by the target vectors.

    Returns
    -------
    np.ndarray
        Array of shape ``(num_traces,)`` with dtype ``np.uint8`` containing
        the attacked AES key byte for every trace.
    """
    key_array = np.asarray(key_bytes, dtype=np.uint8)

    if key_array.ndim == 0:
        return np.full(num_traces, key_array.item(), dtype=np.uint8)

    attacked_key_bytes = _extract_aes_byte(key_array, byte_index)

    if attacked_key_bytes.shape[0] != num_traces:
        raise ValueError("The number of attacked key bytes must match the number of traces.")

    return attacked_key_bytes


def compute_aes_targets_for_poi_selection(plaintext_bytes, key_bytes, byte_index=0):
    """Compute the true AES Hamming-weight targets used only for PoI selection.

    Parameters
    ----------
    plaintext_bytes : np.ndarray
        Plaintext bytes as shape ``(n_traces,)`` or full plaintexts as shape
        ``(n_traces, 16)``.
    key_bytes : int or np.ndarray
        AES key information used on the calibration/profiling traces. This
        can be one fixed attacked byte, one attacked key byte per trace, or
        the full AES key matrix with shape ``(n_traces, 16)``.
    byte_index : int, default=0
        Targeted AES byte index when full plaintexts are provided.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Two arrays of shape ``(n_traces,)``: ``h_m_target`` and ``h_y_target``.
    """
    m_bytes = _extract_aes_byte(plaintext_bytes, byte_index)
    attacked_key_bytes = _prepare_aes_key_bytes(
        key_bytes=key_bytes,
        byte_index=byte_index,
        num_traces=m_bytes.shape[0],
    )
    y_values = aes_sbox(np.bitwise_xor(m_bytes, attacked_key_bytes))

    h_m_target = hamming_weight(m_bytes).astype(np.float64)
    h_y_target = hamming_weight(y_values).astype(np.float64)
    return h_m_target, h_y_target


def find_aes_pois_with_correlation(
    traces,
    plaintext_bytes,
    key_bytes,
    byte_index=0,
    num_pois_per_variable=50,
    min_separation=1,
):
    """Find PoIs for AES by correlating traces with ``HW(m)`` and ``HW(y)``.

    This helper follows the paper's PoI-selection assumption: the attacker uses
    a calibration/profiling set with known public plaintext and known attacked
    key bytes for each trace to locate informative sample points.

    Returns
    -------
    dict
        Dictionary containing target vectors, correlation traces, and selected
        PoI indices for both variables.
    """
    traces = np.asarray(traces, dtype=np.float64)
    h_m_target, h_y_target = compute_aes_targets_for_poi_selection(
        plaintext_bytes=plaintext_bytes,
        key_bytes=key_bytes,
        byte_index=byte_index,
    )

    correlation_h_m = pearson_correlation_to_trace_samples(traces, h_m_target)
    correlation_h_y = pearson_correlation_to_trace_samples(traces, h_y_target)

    pois_h_m = select_top_pois(
        correlation_trace=correlation_h_m,
        num_pois=num_pois_per_variable,
        min_separation=min_separation,
    )
    pois_h_y = select_top_pois(
        correlation_trace=correlation_h_y,
        num_pois=num_pois_per_variable,
        min_separation=min_separation,
    )

    return {
        "h_m_target": h_m_target,
        "h_y_target": h_y_target,
        "correlation_h_m": correlation_h_m,
        "correlation_h_y": correlation_h_y,
        "pois_h_m": pois_h_m,
        "pois_h_y": pois_h_y,
    }


def plot_aes_poi_correlations(correlation_h_m, correlation_h_y, pois_h_m=None, pois_h_y=None):
    """Visualize the correlation traces used for PoI selection.

    If Plotly is available, the returned figure supports zooming and panning
    directly in the notebook. If Plotly is missing from the current kernel,
    the function falls back to Matplotlib so the tutorial cell still runs.
    """
    correlation_h_m = np.asarray(correlation_h_m, dtype=np.float64)
    correlation_h_y = np.asarray(correlation_h_y, dtype=np.float64)

    if correlation_h_m.shape != correlation_h_y.shape:
        raise ValueError("correlation_h_m and correlation_h_y must have the same shape.")

    sample_axis = np.arange(correlation_h_m.size)

    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
    except ImportError:
        print(
            "Plotly is not installed in this Python environment; using a static Matplotlib fallback. "
            "Install `plotly` for zoomable notebook plots."
        )

        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)

        axes[0].plot(sample_axis, correlation_h_m, color="#1f77b4", linewidth=1.2)
        axes[0].set_title("Correlation with h_m = HW(m)")
        axes[0].set_ylabel("Pearson correlation")
        axes[0].grid(alpha=0.25)

        axes[1].plot(sample_axis, correlation_h_y, color="#ff7f0e", linewidth=1.2)
        axes[1].set_title("Correlation with h_y = HW(Sbox(m xor k*))")
        axes[1].set_xlabel("Sample index")
        axes[1].set_ylabel("Pearson correlation")
        axes[1].grid(alpha=0.25)

        if pois_h_m is not None:
            pois_h_m = np.asarray(pois_h_m, dtype=np.int64)
            axes[0].scatter(
                pois_h_m,
                correlation_h_m[pois_h_m],
                color="#00bcd4",
                edgecolor="black",
                s=40,
                zorder=3,
                label="h_m PoIs",
            )
            axes[0].legend(loc="upper right")

        if pois_h_y is not None:
            pois_h_y = np.asarray(pois_h_y, dtype=np.int64)
            axes[1].scatter(
                pois_h_y,
                correlation_h_y[pois_h_y],
                color="#ffd166",
                edgecolor="black",
                s=40,
                zorder=3,
                label="h_y PoIs",
            )
            axes[1].legend(loc="upper right")

        return fig

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.09,
        subplot_titles=(
            "Correlation with h_m = HW(m)",
            "Correlation with h_y = HW(Sbox(m xor k*))",
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=sample_axis,
            y=correlation_h_m,
            mode="lines",
            line=dict(color="#1f77b4", width=1.3),
            name="h_m correlation",
            hovertemplate="sample=%{x}<br>corr=%{y:.4f}<extra>h_m</extra>",
        ),
        row=1,
        col=1,
    )

    if pois_h_m is not None:
        pois_h_m = np.asarray(pois_h_m, dtype=np.int64)
        fig.add_trace(
            go.Scatter(
                x=pois_h_m,
                y=correlation_h_m[pois_h_m],
                mode="markers",
                marker=dict(color="#00bcd4", size=8, line=dict(color="#111111", width=1)),
                name="h_m PoIs",
                hovertemplate="PoI=%{x}<br>corr=%{y:.4f}<extra>h_m PoI</extra>",
            ),
            row=1,
            col=1,
        )

    fig.add_trace(
        go.Scatter(
            x=sample_axis,
            y=correlation_h_y,
            mode="lines",
            line=dict(color="#ff7f0e", width=1.3),
            name="h_y correlation",
            hovertemplate="sample=%{x}<br>corr=%{y:.4f}<extra>h_y</extra>",
        ),
        row=2,
        col=1,
    )

    if pois_h_y is not None:
        pois_h_y = np.asarray(pois_h_y, dtype=np.int64)
        fig.add_trace(
            go.Scatter(
                x=pois_h_y,
                y=correlation_h_y[pois_h_y],
                mode="markers",
                marker=dict(color="#ffd166", size=8, line=dict(color="#111111", width=1)),
                name="h_y PoIs",
                hovertemplate="PoI=%{x}<br>corr=%{y:.4f}<extra>h_y PoI</extra>",
            ),
            row=2,
            col=1,
        )

    fig.update_yaxes(title_text="Pearson correlation", row=1, col=1)
    fig.update_yaxes(title_text="Pearson correlation", row=2, col=1)
    fig.update_xaxes(title_text="Sample index", row=2, col=1)
    fig.update_layout(
        height=820,
        width=1200,
        template="plotly_white",
        hovermode="x unified",
        dragmode="zoom",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
        margin=dict(l=70, r=30, t=90, b=60),
    )

    return fig


def display_notebook_figure(fig):
    """Display Plotly or Matplotlib figures robustly inside notebooks.

    Plotly's default notebook renderer can fail when ``nbformat`` is missing
    from the kernel environment. To keep the tutorial runnable, Plotly
    figures are rendered as HTML explicitly, while Matplotlib figures are
    displayed through IPython's standard display hook.
    """
    try:
        from IPython.display import HTML, display
    except ImportError:
        if hasattr(fig, "show"):
            fig.show()
        return

    if hasattr(fig, "to_html"):
        # Inline plotly.js so the figure renders on the first execution of
        # the cell instead of depending on a prior CDN script load.
        display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))
    else:
        display(fig)

### Expected Inputs for PoI Selection

The helper above expects:

- `traces`: a NumPy array with shape `(n_traces, n_samples)`
- `plaintext_bytes`: either shape `(n_traces,)` for one attacked byte per trace, or `(n_traces, 16)` for full AES plaintexts
- `key_bytes`: the known AES key information for the profiling traces; this may be one attacked byte per trace or a full `(n_traces, 16)` key matrix
- `byte_index`: the attacked AES byte position

Below is a template cell you can fill in once the traces and metadata are loaded.

## Download Dataset

If you are running this notebook in Google Colab, download and extract the dataset archive before executing the next cell:

- Download URL: `https://github.com/Zer0Leak/CISSA/releases/download/v00.01/Datasets.zip`
- The ZIP extracts a `Datasets/Chipwhisperer/` directory used by the cells below.
- After unzipping, the archive provides these NumPy files:
  - `Datasets/Chipwhisperer/Desync_traces_10.npy`: desynchronized trace matrix used in the PoI-selection example below
  - `Datasets/Chipwhisperer/traces.npy`: synchronized ChipWhisperer trace matrix
  - `Datasets/Chipwhisperer/plain.npy`: plaintext bytes for each trace
  - `Datasets/Chipwhisperer/key.npy`: key bytes aligned with the traces
  - `Datasets/Chipwhisperer/labels.npy`: labels provided with the dataset

Run the next code cell in Colab to download and unzip the archive only when the expected dataset files are not already present. The cell also verifies `Datasets.zip` with the expected MD5 checksum before reusing it.


In [ ]:
%%bash
set -euo pipefail

dataset_url="https://github.com/Zer0Leak/CISSA/releases/download/v00.01/Datasets.zip"
zip_path="Datasets.zip"
dataset_dir="Datasets/Chipwhisperer"
expected_md5="65a166b0532f9ce7cfa723f8139b3c07"

expected_files=(
  "$dataset_dir/Desync_traces_10.npy"
  "$dataset_dir/traces.npy"
  "$dataset_dir/plain.npy"
  "$dataset_dir/key.npy"
  "$dataset_dir/labels.npy"
)

compute_md5() {
  md5sum "$1" | awk '{print $1}'
}

all_files_present=1
for file_path in "${expected_files[@]}"; do
  if [ ! -f "$file_path" ]; then
    all_files_present=0
    break
  fi
done

if [ "$all_files_present" -eq 1 ]; then
  echo "Dataset files already exist. Skipping download and extraction."
else
  zip_is_valid=0
  if [ -f "$zip_path" ]; then
    actual_md5=$(compute_md5 "$zip_path")
    if [ "$actual_md5" = "$expected_md5" ] && unzip -tq "$zip_path" >/dev/null 2>&1; then
      zip_is_valid=1
    else
      echo "Existing $zip_path failed validation. Re-downloading archive."
      rm -f "$zip_path"
    fi
  fi

  if [ "$zip_is_valid" -eq 1 ]; then
    echo "Reusing existing $zip_path with matching MD5 checksum."
  else
    echo "Downloading dataset archive to $zip_path ..."
    wget -q "$dataset_url" -O "$zip_path"
    actual_md5=$(compute_md5 "$zip_path")
    if [ "$actual_md5" != "$expected_md5" ]; then
      echo "MD5 mismatch for $zip_path: expected $expected_md5, got $actual_md5" >&2
      exit 1
    fi
  fi

  unzip -qo "$zip_path"
  echo "Dataset ready in $dataset_dir/."
fi


In [ ]:
from pathlib import Path

# Load the desynchronized ChipWhisperer AES traces together with the matching
# plaintexts and keys used for PoI selection.
traces_path = Path("Datasets/Chipwhisperer/Desync_traces_10.npy")
plaintext_path = Path("Datasets/Chipwhisperer/plain.npy")
key_path = Path("Datasets/Chipwhisperer/key.npy")

profiling_traces = np.load(traces_path)
profiling_plaintexts = np.load(plaintext_path)
profiling_keys = np.load(key_path)

# The dataset files do not have exactly the same number of rows, so we keep
# only the common prefix shared by traces, plaintexts, and keys.
num_trace_rows = profiling_traces.shape[0]
num_plaintext_rows = profiling_plaintexts.shape[0]
num_key_rows = profiling_keys.shape[0]

if len({num_trace_rows, num_plaintext_rows, num_key_rows}) != 1:
    common_num_traces = min(num_trace_rows, num_plaintext_rows, num_key_rows)
    print(
        "Warning: traces, plaintext, and key arrays have different lengths. "
        f"Using the first {common_num_traces} rows from each array."
    )
    profiling_traces = profiling_traces[:common_num_traces]
    profiling_plaintexts = profiling_plaintexts[:common_num_traces]
    profiling_keys = profiling_keys[:common_num_traces]

print("profiling_traces shape:", profiling_traces.shape)
print("profiling_plaintexts shape:", profiling_plaintexts.shape)
print("profiling_keys shape:", profiling_keys.shape)

# Keep the full key matrix. The PoI helper accepts one attacked key byte per
# trace, so we do not assume a single fixed AES key across the profiling set.
print("example profiling key row:", " ".join(f"0x{x:02X}" for x in profiling_keys[0]))


In [ ]:
# Fill in the attacked byte index. The helper will extract the matching key
# byte from every row of profiling_keys, so no fixed-key assumption is made.

poi_results = []

for byte_index in range(16):
    poi_result = find_aes_pois_with_correlation(
        traces=profiling_traces,
        plaintext_bytes=profiling_plaintexts,
        key_bytes=profiling_keys,
        byte_index=byte_index,
        num_pois_per_variable=50,
        min_separation=1,
    )
    poi_results.append(poi_result)


In [ ]:

# We keep byte 8 as the running example for the next steps of the tutorial.

attacked_byte_index = 11
poi_result = poi_results[attacked_byte_index]

print(f"Showing PoIs for byte index {attacked_byte_index}")

print("PoIs for h_m:", poi_result["pois_h_m"])
print("PoIs for h_y:", poi_result["pois_h_y"])

fig = plot_aes_poi_correlations(
    correlation_h_m=poi_result["correlation_h_m"],
    correlation_h_y=poi_result["correlation_h_y"],
    pois_h_m=poi_result["pois_h_m"],
    pois_h_y=poi_result["pois_h_y"],
)
display_notebook_figure(fig)

# Step 4: Build Labels from Those PoIs

We now move from **PoI selection** to **MC labeling** (*Multi-point Cluster-based labeling*), the label-construction technique proposed in the paper.

For the selected byte, each trace is truncated to the PoIs found in Step 3:

- 50 samples from `PoIm`, the PoIs correlated with `h_m = HW(m)`
- 50 samples from `PoIy`, the PoIs correlated with `h_y = HW(Sbox(m xor k*))`

So each original trace becomes a compact 100-dimensional vector. Then the idea is:

1. Cluster those 100-dimensional truncated traces into `9 x 9 = 81` groups, because `h_m` and `h_y` can each take values in `{0, ..., 8}`.
2. Compute the center trace of each cluster.
3. For each PoI position, apply **slicing labeling** across the cluster centers, which gives one candidate Hamming-weight label per PoI.
4. Combine those candidate labels with the paper's **weighted majority vote**, using weights `1 / binom(8, h)`.
5. Assign the final cluster label `(h_m, h_y)` to every trace in that cluster.

Two practical details matter here:

- Some PoIs have **negative** correlation. Before slicing, we flip the sign of those PoIs so that larger aligned amplitude consistently suggests larger Hamming weight.
- In a truly blind setting, we would not know the real `(h_m, h_y)` labels. In this notebook we still compute the ground truth from plaintext and key **only to evaluate how noisy the constructed labels are**.


In [ ]:
from math import comb

try:
    from sklearn.decomposition import PCA
    from sklearn.mixture import GaussianMixture
    from sklearn.preprocessing import StandardScaler
except ImportError as exc:
    raise ImportError(
        "Step 4 requires scikit-learn. In Google Colab it is usually preinstalled. "
        "If you run locally, install it with `pip install scikit-learn`."
    ) from exc


def binomial_hw_probabilities(num_bits=8):
    """Return Pr(HW(x) = h) for a uniformly random ``num_bits``-bit value."""
    return np.array(
        [comb(num_bits, h) for h in range(num_bits + 1)],
        dtype=np.float64,
    ) / (2 ** num_bits)


def allocate_slicing_counts(num_items, num_bits=8):
    """Allocate how many sorted samples should receive each HW label."""
    probabilities = binomial_hw_probabilities(num_bits=num_bits)
    raw_counts = num_items * probabilities
    class_counts = np.floor(raw_counts).astype(np.int64)

    remainder = int(num_items - class_counts.sum())
    if remainder > 0:
        order = np.argsort(raw_counts - class_counts)[::-1]
        class_counts[order[:remainder]] += 1

    return class_counts


def slicing_label_1d(values, num_bits=8):
    """Apply the paper's slicing labeling to one scalar feature."""
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    class_counts = allocate_slicing_counts(num_items=values.size, num_bits=num_bits)
    sort_order = np.argsort(values)
    labels = np.empty(values.size, dtype=np.int64)

    start = 0
    for hw_label, class_count in enumerate(class_counts):
        if class_count == 0:
            continue
        stop = start + int(class_count)
        labels[sort_order[start:stop]] = hw_label
        start = stop

    return labels, class_counts


def extract_mc_features_from_aes_pois(traces, poi_result, align_signs=True):
    """Truncate each trace to the selected PoIs for ``h_m`` and ``h_y``."""
    traces = np.asarray(traces, dtype=np.float64)

    pois_h_m = np.asarray(poi_result["pois_h_m"], dtype=np.int64)
    pois_h_y = np.asarray(poi_result["pois_h_y"], dtype=np.int64)

    features_h_m = traces[:, pois_h_m].copy()
    features_h_y = traces[:, pois_h_y].copy()

    sign_h_m = np.sign(np.asarray(poi_result["correlation_h_m"], dtype=np.float64)[pois_h_m])
    sign_h_y = np.sign(np.asarray(poi_result["correlation_h_y"], dtype=np.float64)[pois_h_y])
    sign_h_m[sign_h_m == 0.0] = 1.0
    sign_h_y[sign_h_y == 0.0] = 1.0

    if align_signs:
        features_h_m *= sign_h_m
        features_h_y *= sign_h_y

    feature_matrix = np.concatenate([features_h_m, features_h_y], axis=1)
    feature_info = {
        "pois_h_m": pois_h_m,
        "pois_h_y": pois_h_y,
        "sign_h_m": sign_h_m,
        "sign_h_y": sign_h_y,
        "num_pois_h_m": features_h_m.shape[1],
        "num_pois_h_y": features_h_y.shape[1],
    }
    return feature_matrix, feature_info


def compute_cluster_centers(feature_matrix, cluster_assignments, num_clusters, fallback_centers=None):
    """Average the traces inside each cluster to build its center trace."""
    feature_matrix = np.asarray(feature_matrix, dtype=np.float64)
    cluster_assignments = np.asarray(cluster_assignments, dtype=np.int64).reshape(-1)

    cluster_sizes = np.bincount(cluster_assignments, minlength=num_clusters)
    centers = np.zeros((num_clusters, feature_matrix.shape[1]), dtype=np.float64)

    for cluster_index in range(num_clusters):
        in_cluster = cluster_assignments == cluster_index
        if np.any(in_cluster):
            centers[cluster_index] = feature_matrix[in_cluster].mean(axis=0)
        elif fallback_centers is not None:
            centers[cluster_index] = fallback_centers[cluster_index]

    return centers, cluster_sizes


def weighted_majority_vote(candidate_labels, num_bits=8):
    """Aggregate per-PoI candidate labels using the paper's weighted vote."""
    candidate_labels = np.asarray(candidate_labels, dtype=np.int64)
    num_classes = num_bits + 1
    weighted_scores = np.zeros((candidate_labels.shape[0], num_classes), dtype=np.float64)

    for hw_label in range(num_classes):
        vote_count = np.sum(candidate_labels == hw_label, axis=1)
        weighted_scores[:, hw_label] = vote_count / comb(num_bits, hw_label)

    final_labels = np.argmax(weighted_scores, axis=1).astype(np.int64)
    return final_labels, weighted_scores


def build_mc_labels_from_pois(
    feature_matrix,
    num_pois_h_m,
    num_bits=8,
    covariance_type="diag",
    random_state=0,
    max_iter=100,
):
    """Implement the paper's MC labeling on truncated PoI traces."""
    feature_matrix = np.asarray(feature_matrix, dtype=np.float64)
    num_clusters = (num_bits + 1) ** 2

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(feature_matrix)

    gmm = GaussianMixture(
        n_components=num_clusters,
        covariance_type=covariance_type,
        reg_covar=1e-6,
        n_init=1,
        max_iter=max_iter,
        random_state=random_state,
    )
    cluster_assignments = gmm.fit_predict(scaled_features)

    cluster_centers, cluster_sizes = compute_cluster_centers(
        feature_matrix=scaled_features,
        cluster_assignments=cluster_assignments,
        num_clusters=num_clusters,
        fallback_centers=getattr(gmm, "means_", None),
    )

    cluster_centers_h_m = cluster_centers[:, :num_pois_h_m]
    cluster_centers_h_y = cluster_centers[:, num_pois_h_m:]

    candidate_labels_h_m = np.column_stack(
        [
            slicing_label_1d(cluster_centers_h_m[:, poi_index], num_bits=num_bits)[0]
            for poi_index in range(cluster_centers_h_m.shape[1])
        ]
    )
    candidate_labels_h_y = np.column_stack(
        [
            slicing_label_1d(cluster_centers_h_y[:, poi_index], num_bits=num_bits)[0]
            for poi_index in range(cluster_centers_h_y.shape[1])
        ]
    )

    cluster_labels_h_m, vote_scores_h_m = weighted_majority_vote(
        candidate_labels=candidate_labels_h_m,
        num_bits=num_bits,
    )
    cluster_labels_h_y, vote_scores_h_y = weighted_majority_vote(
        candidate_labels=candidate_labels_h_y,
        num_bits=num_bits,
    )

    trace_labels_h_m = cluster_labels_h_m[cluster_assignments]
    trace_labels_h_y = cluster_labels_h_y[cluster_assignments]

    return {
        "feature_matrix": feature_matrix,
        "scaled_features": scaled_features,
        "gmm": gmm,
        "num_clusters": num_clusters,
        "cluster_assignments": cluster_assignments,
        "cluster_sizes": cluster_sizes,
        "cluster_centers": cluster_centers,
        "candidate_labels_h_m": candidate_labels_h_m,
        "candidate_labels_h_y": candidate_labels_h_y,
        "vote_scores_h_m": vote_scores_h_m,
        "vote_scores_h_y": vote_scores_h_y,
        "cluster_labels_h_m": cluster_labels_h_m,
        "cluster_labels_h_y": cluster_labels_h_y,
        "trace_labels_h_m": trace_labels_h_m,
        "trace_labels_h_y": trace_labels_h_y,
    }


def compute_confusion_matrix(true_labels, predicted_labels, num_classes):
    """Return a confusion matrix with true labels on rows."""
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(matrix, (true_labels, predicted_labels), 1)
    return matrix


def plot_mc_pca_projection(scaled_features, predicted_h_m, predicted_h_y, num_examples=2500):
    """Project PoI-truncated traces to 2D so the clusters become visible."""
    scaled_features = np.asarray(scaled_features, dtype=np.float64)
    predicted_h_m = np.asarray(predicted_h_m, dtype=np.int64)
    predicted_h_y = np.asarray(predicted_h_y, dtype=np.int64)

    num_examples = min(num_examples, scaled_features.shape[0])
    pca = PCA(n_components=2)
    projected = pca.fit_transform(scaled_features[:num_examples])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

    scatter = axes[0].scatter(
        projected[:, 0],
        projected[:, 1],
        c=predicted_h_m[:num_examples],
        cmap="viridis",
        s=12,
        alpha=0.75,
    )
    axes[0].set_title("MC labels projected to 2D\ncolored by predicted h_m")
    axes[0].set_xlabel("PC 1")
    axes[0].set_ylabel("PC 2")
    colorbar = fig.colorbar(scatter, ax=axes[0], shrink=0.85)
    colorbar.set_label("predicted h_m")

    scatter = axes[1].scatter(
        projected[:, 0],
        projected[:, 1],
        c=predicted_h_y[:num_examples],
        cmap="plasma",
        s=12,
        alpha=0.75,
    )
    axes[1].set_title("Same projection\ncolored by predicted h_y")
    axes[1].set_xlabel("PC 1")
    axes[1].set_ylabel("PC 2")
    colorbar = fig.colorbar(scatter, ax=axes[1], shrink=0.85)
    colorbar.set_label("predicted h_y")

    return fig, axes


def plot_mc_feature_heatmap(scaled_features, predicted_h_m, predicted_h_y, num_pois_h_m, num_examples=160):
    """Show how the PoI-truncated traces look after sorting by the MC labels."""
    scaled_features = np.asarray(scaled_features, dtype=np.float64)
    predicted_h_m = np.asarray(predicted_h_m, dtype=np.int64)
    predicted_h_y = np.asarray(predicted_h_y, dtype=np.int64)

    sort_order = np.lexsort((predicted_h_y, predicted_h_m))
    num_examples = min(num_examples, scaled_features.shape[0])
    selected_indices = sort_order[:num_examples]
    heatmap = scaled_features[selected_indices]

    fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
    image = ax.imshow(heatmap, aspect="auto", cmap="coolwarm", interpolation="nearest")
    ax.axvline(num_pois_h_m - 0.5, color="black", linewidth=2)
    ax.set_title("Truncated traces after PoI selection\nrows sorted by predicted (h_m, h_y)")
    ax.set_xlabel("PoI index inside the truncated trace")
    ax.set_ylabel("Trace index")
    ax.text(0.25, 1.02, "PoIs for h_m", transform=ax.transAxes, ha="center", va="bottom", fontsize=11)
    ax.text(0.75, 1.02, "PoIs for h_y", transform=ax.transAxes, ha="center", va="bottom", fontsize=11)
    colorbar = fig.colorbar(image, ax=ax, shrink=0.85)
    colorbar.set_label("standardized sample value")

    return fig, ax


def plot_mc_label_diagnostics(
    true_h_m,
    predicted_h_m,
    true_h_y,
    predicted_h_y,
    vote_scores_h_m,
    vote_scores_h_y,
    cluster_sizes,
    cluster_labels_h_m,
    cluster_labels_h_y,
    cluster_index,
):
    """Summarize label noise and one cluster's weighted vote."""
    true_h_m = np.asarray(true_h_m, dtype=np.int64)
    predicted_h_m = np.asarray(predicted_h_m, dtype=np.int64)
    true_h_y = np.asarray(true_h_y, dtype=np.int64)
    predicted_h_y = np.asarray(predicted_h_y, dtype=np.int64)

    confusion_h_m = compute_confusion_matrix(true_h_m, predicted_h_m, num_classes=9)
    confusion_h_y = compute_confusion_matrix(true_h_y, predicted_h_y, num_classes=9)

    def normalize_rows(matrix):
        row_sums = matrix.sum(axis=1, keepdims=True)
        return np.divide(
            matrix,
            row_sums,
            out=np.zeros_like(matrix, dtype=np.float64),
            where=row_sums > 0,
        )

    normalized_h_m = normalize_rows(confusion_h_m)
    normalized_h_y = normalize_rows(confusion_h_y)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    image = axes[0, 0].imshow(normalized_h_m, vmin=0.0, vmax=1.0, cmap="Blues")
    axes[0, 0].set_title("Confusion matrix for h_m\n(rows = true, columns = MC label)")
    axes[0, 0].set_xlabel("Predicted h_m")
    axes[0, 0].set_ylabel("True h_m")
    axes[0, 0].set_xticks(range(9))
    axes[0, 0].set_yticks(range(9))
    fig.colorbar(image, ax=axes[0, 0], shrink=0.85)

    image = axes[0, 1].imshow(normalized_h_y, vmin=0.0, vmax=1.0, cmap="Oranges")
    axes[0, 1].set_title("Confusion matrix for h_y\n(rows = true, columns = MC label)")
    axes[0, 1].set_xlabel("Predicted h_y")
    axes[0, 1].set_ylabel("True h_y")
    axes[0, 1].set_xticks(range(9))
    axes[0, 1].set_yticks(range(9))
    fig.colorbar(image, ax=axes[0, 1], shrink=0.85)

    axes[1, 0].bar(range(9), vote_scores_h_m[cluster_index], color="#1f77b4")
    axes[1, 0].set_title(
        f"Weighted votes for one cluster on h_m\ncluster={cluster_index}, size={cluster_sizes[cluster_index]}, chosen label={cluster_labels_h_m[cluster_index]}"
    )
    axes[1, 0].set_xlabel("Candidate h_m")
    axes[1, 0].set_ylabel("Weighted vote score")
    axes[1, 0].set_xticks(range(9))

    axes[1, 1].bar(range(9), vote_scores_h_y[cluster_index], color="#ff7f0e")
    axes[1, 1].set_title(
        f"Weighted votes for the same cluster on h_y\ncluster={cluster_index}, size={cluster_sizes[cluster_index]}, chosen label={cluster_labels_h_y[cluster_index]}"
    )
    axes[1, 1].set_xlabel("Candidate h_y")
    axes[1, 1].set_ylabel("Weighted vote score")
    axes[1, 1].set_xticks(range(9))

    return fig, axes


In [ ]:
# Keep using the same byte that we visualized at the end of Step 3.
# For tutorial speed, we start with a subset of the profiling traces.
# You can increase this later once the pipeline is working as expected.

poi_result = poi_results[attacked_byte_index]
mc_labeling_num_traces = min(3000, profiling_traces.shape[0])

mc_traces = profiling_traces[:mc_labeling_num_traces]
mc_plaintexts = profiling_plaintexts[:mc_labeling_num_traces]
mc_keys = profiling_keys[:mc_labeling_num_traces]

mc_feature_matrix, mc_feature_info = extract_mc_features_from_aes_pois(
    traces=mc_traces,
    poi_result=poi_result,
    align_signs=True,
)

mc_result = build_mc_labels_from_pois(
    feature_matrix=mc_feature_matrix,
    num_pois_h_m=mc_feature_info["num_pois_h_m"],
    num_bits=8,
    covariance_type="diag",
    random_state=0,
    max_iter=100,
)

# In a real blind setting these true labels would not be available.
# We compute them here only to inspect how noisy the MC labels are.
true_h_m, true_h_y = compute_aes_targets_for_poi_selection(
    plaintext_bytes=mc_plaintexts,
    key_bytes=mc_keys,
    byte_index=attacked_byte_index,
)
true_h_m = true_h_m.astype(np.int64)
true_h_y = true_h_y.astype(np.int64)

mc_accuracy_h_m = np.mean(mc_result["trace_labels_h_m"] == true_h_m)
mc_accuracy_h_y = np.mean(mc_result["trace_labels_h_y"] == true_h_y)
mc_joint_accuracy = np.mean(
    (mc_result["trace_labels_h_m"] == true_h_m)
    & (mc_result["trace_labels_h_y"] == true_h_y)
)

mc_trace_labels = np.column_stack(
    [mc_result["trace_labels_h_m"], mc_result["trace_labels_h_y"]]
)

print(f"Byte under study: {attacked_byte_index}")
print("MC feature matrix shape:", mc_feature_matrix.shape)
print("GMM converged:", mc_result["gmm"].converged_)
print("Number of GMM clusters:", mc_result["num_clusters"])
print(
    "Cluster size range:",
    int(mc_result["cluster_sizes"].min()),
    "to",
    int(mc_result["cluster_sizes"].max()),
)
print(f"Evaluation-only accuracy for h_m: {mc_accuracy_h_m:.3f}")
print(f"Evaluation-only accuracy for h_y: {mc_accuracy_h_y:.3f}")
print(f"Evaluation-only joint accuracy for (h_m, h_y): {mc_joint_accuracy:.3f}")
print("First 10 MC labels [h_m, h_y]:")
print(mc_trace_labels[:10])

fig, axes = plot_mc_pca_projection(
    scaled_features=mc_result["scaled_features"],
    predicted_h_m=mc_result["trace_labels_h_m"],
    predicted_h_y=mc_result["trace_labels_h_y"],
    num_examples=2500,
)
plt.show()

fig, ax = plot_mc_feature_heatmap(
    scaled_features=mc_result["scaled_features"],
    predicted_h_m=mc_result["trace_labels_h_m"],
    predicted_h_y=mc_result["trace_labels_h_y"],
    num_pois_h_m=mc_feature_info["num_pois_h_m"],
    num_examples=160,
)
plt.show()

example_cluster_index = int(np.argmax(mc_result["cluster_sizes"]))
fig, axes = plot_mc_label_diagnostics(
    true_h_m=true_h_m,
    predicted_h_m=mc_result["trace_labels_h_m"],
    true_h_y=true_h_y,
    predicted_h_y=mc_result["trace_labels_h_y"],
    vote_scores_h_m=mc_result["vote_scores_h_m"],
    vote_scores_h_y=mc_result["vote_scores_h_y"],
    cluster_sizes=mc_result["cluster_sizes"],
    cluster_labels_h_m=mc_result["cluster_labels_h_m"],
    cluster_labels_h_y=mc_result["cluster_labels_h_y"],
    cluster_index=example_cluster_index,
)
plt.show()
